<h1>PKL->PNG

<h5>跑前3個就可以了，或是你可以來實驗室copy圖片過去，他檔案很大




In [1]:
import pickle

with open(r"C:\Users\user\Desktop\MIR-WM811K\MIR-WM811K\Python\WM811K.pkl", "rb") as f:
    data = pickle.load(f)

print(type(data))
print(data)


C:\Users\user\AppData\Local\Temp\ipykernel_21740\2589417509.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


<class 'pandas.core.frame.DataFrame'>
        dieSize failureType   lotName trainTestLabel  waferIndex  \
0        1683.0        none      lot1       Training         1.0   
1        1683.0        none      lot1       Training         2.0   
2        1683.0        none      lot1       Training         3.0   
3        1683.0        none      lot1       Training         4.0   
4        1683.0        none      lot1       Training         5.0   
...         ...         ...       ...            ...         ...   
811452    600.0   Edge-Ring  lot47542           Test        23.0   
811453    600.0    Edge-Loc  lot47542           Test        24.0   
811454    600.0   Edge-Ring  lot47542           Test        25.0   
811455    600.0      [0, 0]  lot47543         [0, 0]         1.0   
811456    600.0      [0, 0]  lot47543         [0, 0]         2.0   

                                                 waferMap  
0       [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...  
1       [[0, 0, 0, 0, 0, 

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import random

# ==========================
# 1️⃣ 讀取 WM811K.pkl
# ==========================
with open(r"C:\Users\user\Desktop\MIR-WM811K\MIR-WM811K\Python\WM811K.pkl", "rb") as f:
    data = pickle.load(f)

print("資料欄位:", data.keys())

wafer_maps = data["waferMap"]
failure_types = data["failureType"]

print(f"總 wafer 數量: {len(wafer_maps)}")

# ==========================
# 2️⃣ 13 區計算函式
# ==========================
def cal_den(x):
    """計算 defect 密度 (值=2 的比例)"""
    return 100*(np.sum(x==2)/np.size(x))

def find_regions_arrays(x):
    """把 wafer map 分成 13 區塊"""
    rows = np.size(x, axis=0)
    cols = np.size(x, axis=1)
    ind1 = np.arange(0, rows, rows//5)
    ind2 = np.arange(0, cols, cols//5)

    reg1  = x[ind1[0]:ind1[1], :]
    reg3  = x[ind1[4]:, :]
    reg4  = x[:, ind2[0]:ind2[1]]
    reg2  = x[:, ind2[4]:]

    reg5  = x[ind1[1]:ind1[2], ind2[1]:ind2[2]]
    reg6  = x[ind1[1]:ind1[2], ind2[2]:ind2[3]]
    reg7  = x[ind1[1]:ind1[2], ind2[3]:ind2[4]]
    reg8  = x[ind1[2]:ind1[3], ind2[1]:ind2[2]]
    reg9  = x[ind1[2]:ind1[3], ind2[2]:ind2[3]]
    reg10 = x[ind1[2]:ind1[3], ind2[3]:ind2[4]]
    reg11 = x[ind1[3]:ind1[4], ind2[1]:ind2[2]]
    reg12 = x[ind1[3]:ind1[4], ind2[2]:ind2[3]]
    reg13 = x[ind1[3]:ind1[4], ind2[3]:ind2[4]]

    return [
        reg1, reg2, reg3, reg4, reg5, reg6, reg7,
        reg8, reg9, reg10, reg11, reg12, reg13
    ]

# ==========================
# 3️⃣ 彩色視覺化 13 區
# ==========================
def visualize_13_regions(wm):
    regions = find_regions_arrays(wm)
    h, w = wm.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)

    rows = np.size(wm, axis=0)
    cols = np.size(wm, axis=1)
    ind1 = np.arange(0, rows, rows//5)
    ind2 = np.arange(0, cols, cols//5)

    coords = [
        (slice(ind1[0], ind1[1]), slice(None)),               # reg1
        (slice(None), slice(ind2[4], None)),                 # reg2
        (slice(ind1[4], None), slice(None)),                 # reg3
        (slice(None), slice(ind2[0], ind2[1])),              # reg4
        (slice(ind1[1], ind1[2]), slice(ind2[1], ind2[2])),  # reg5
        (slice(ind1[1], ind1[2]), slice(ind2[2], ind2[3])),  # reg6
        (slice(ind1[1], ind1[2]), slice(ind2[3], ind2[4])),  # reg7
        (slice(ind1[2], ind1[3]), slice(ind2[1], ind2[2])),  # reg8
        (slice(ind1[2], ind1[3]), slice(ind2[2], ind2[3])),  # reg9
        (slice(ind1[2], ind1[3]), slice(ind2[3], ind2[4])),  # reg10
        (slice(ind1[3], ind1[4]), slice(ind2[1], ind2[2])),  # reg11
        (slice(ind1[3], ind1[4]), slice(ind2[2], ind2[3])),  # reg12
        (slice(ind1[3], ind1[4]), slice(ind2[3], ind2[4])),  # reg13
    ]

    for i, c in enumerate(coords):
        color_map[c] = ((i*17) % 256, (i*40) % 256, (i*70) % 256)

    plt.figure(figsize=(6,6))
    plt.title("13 Region Segmentation")
    plt.imshow(color_map)
    plt.axis("off")
    plt.show()

    print("=== 13 區 defect density (值=2 的比例 %) ===")
    for idx, reg in enumerate(regions, start=1):
        den = cal_den(reg)
        print(f"Region {idx:02d}: {den:.2f} % ")

# ==========================
# 4️⃣ 隨機挑一張 wafer 做示範
# ==========================
idx = random.randint(0, len(wafer_maps)-1)
wm = wafer_maps[idx]
print(f"挑選第 {idx} 張 wafer，failure type = {failure_types[idx]}")

visualize_13_regions(wm)


In [ ]:
import pandas as pd

# 讀取 PKL
df = pd.read_pickle('WM811K.pkl')  # 改成你的檔案路徑

# 取前 1000 筆
df_head = df.head(100)

# 轉存成 CSV
df_head.to_csv('output_1000.csv', index=False)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import numpy as np

y_true = test_set.classes
y_pred_prob = model.predict(test_set, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=train_set.class_indices.keys(),
            yticklabels=train_set.class_indices.keys(), cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
print(data.keys())


In [ ]:
import pickle
import numpy as np
from PIL import Image
import os

# ===========================
# 讀取 PKL
# ===========================
with open("WM811K.pkl", "rb") as f:
    data = pickle.load(f)

wafer_maps = data["waferMap"]
failure_types = data["failureType"]

print("Total wafers =", len(wafer_maps))

# ===========================
# 建立主資料夾
# ===========================
output_root = "wafer_images_fast"
os.makedirs(output_root, exist_ok=True)

# ===========================
# 顏色對應表（0~9）
# ===========================
color_lut = np.zeros((10, 3), dtype=np.uint8)
color_lut[0] = (0, 0, 0)        # background
color_lut[1] = (0, 200, 0)      # normal die
color_lut[2:] = (255, 0, 0)     # all defect dies (2~9)

# ===========================
# 迭代每張 wafer map（超快）
# ===========================
for idx, (wm, label) in enumerate(zip(wafer_maps, failure_types)):

    # 建 label 資料夾
    type_dir = os.path.join(output_root, f"type_{label}")
    os.makedirs(type_dir, exist_ok=True)

    # ---- 直接查 LUT，最效率 ----
    # wm = 2D array，shape (h, w)
    rgb = color_lut[wm]  # ★ numpy 一次完成，不用 for-loop

    # 轉成圖片
    img = Image.fromarray(rgb.astype(np.uint8))

    # 儲存
    filename = f"wafer_{idx:06d}.png"
    img.save(os.path.join(type_dir, filename))

    # 進度視窗
    if idx % 2000 == 0:
        print(f"Saved {idx}/{len(wafer_maps)}")

print("全部完成！(終極加速版本)")


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

with open(r"C:\Users\user\Desktop\MIR-WM811K\MIR-WM811K\Python\WM811K.pkl", "rb") as f:
    data = pickle.load(f)

# 假設 data 是 dict
wafer_maps = data["waferMap"]

# 取出第一張 wafer map
wm = wafer_maps[0]  # 可能還是 list of list

# 轉成 numpy array
wm_np = np.array(wm, dtype=np.uint8)  # 或 float, 根據你的需求

# 畫圖
plt.imshow(wm_np, cmap='gray')
plt.axis('off')
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 取第一筆
img = np.array(data.loc[0, "waferMap"])

plt.imshow(img)
plt.title(f"failureType = {data.loc[0,'failureType']}")
plt.colorbar()
plt.show()


In [ ]:
idx = 1000
img = np.array(data.loc[idx, "waferMap"])

plt.imshow(img)
plt.title(f"failureType = {data.loc[idx,'failureType']}")
plt.colorbar()
plt.show()


In [ ]:
 #illustration of 13 regions
an = np.linspace(0, 2*np.pi, 100)
plt.plot(2.5*np.cos(an), 2.5*np.sin(an))
plt.axis('equal')
plt.axis([-4, 4, -4, 4])
plt.plot([-2.5, 2.5], [1.5, 1.5])
plt.plot([-2.5, 2.5], [0.5, 0.5 ])
plt.plot([-2.5, 2.5], [-0.5, -0.5 ])
plt.plot([-2.5, 2.5], [-1.5,-1.5 ])

plt.plot([0.5, 0.5], [-2.5, 2.5])
plt.plot([1.5, 1.5], [-2.5, 2.5])
plt.plot([-0.5, -0.5], [-2.5, 2.5])
plt.plot([-1.5, -1.5], [-2.5, 2.5])
plt.title(" Devide wafer map to 13 regions")
plt.xticks([])
plt.yticks([])
plt.show()